# Retos: crea tu propio chatbot

**Antes de empezar:** configura `OPENROUTER_API_KEY` en tu archivo `.env`, como en los notebooks anteriores. Nunca escribas la clave en una celda ni la compartas.

In [ ]:
import os

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "https://openrouter.ai/api/v1"
MODELO_FREE = "meta-llama/llama-3.3-70b-instruct:free"
API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert API_KEY, "Falta OPENROUTER_API_KEY en el archivo .env"

def preguntar(messages: list[dict], modelo: str = MODELO_FREE) -> str:
    """Envía la conversación y devuelve el texto de la respuesta."""
    respuesta = httpx.post(
        f"{BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": modelo, "messages": messages},
        timeout=60,
    )
    respuesta.raise_for_status()
    return respuesta.json()["choices"][0]["message"]["content"]

## Ejercicio 1: dale una personalidad

El mensaje `system` le da instrucciones al asistente. Cambia el rol y las reglas para crear un personaje útil: por ejemplo, un entrenador de Python que explica con ejemplos cortos y hace una pregunta para comprobar que entendiste.

1. Ejecuta la celda tal como está.
2. Cambia las instrucciones y vuelve a ejecutarla con la misma pregunta.
3. Prueba qué pasa si pides una respuesta de una sola frase o una explicación para alguien que empieza.

**Comprueba:** ¿qué instrucciones cambiaron más la respuesta?

In [ ]:
messages = [
    {"role": "system", "content": "Eres un tutor de programación paciente. Explica con un ejemplo sencillo."},
    {"role": "user", "content": "¿Para qué sirve un bucle?"},
]
respuesta = preguntar(messages)
print("bot>", respuesta)

## Ejercicio 2: construye una conversación

La API no recuerda llamadas anteriores. El historial vive en `messages`, así que hay que enviar la lista completa cada vez. Completa los tres pasos marcados en la celda:

1. Guarda lo que escribió la persona como un mensaje con rol `user`.
2. Llama a `preguntar(messages)` y guarda la respuesta.
3. Imprime la respuesta y añádela al historial con rol `assistant`.

Escribe `salir` o `exit` para terminar. Cuando funcione, pregúntale algo y luego haz una pregunta de seguimiento como «¿puedes darme un ejemplo?».

In [ ]:
messages = [
    {"role": "system", "content": "Eres un asistente breve y claro que ayuda a aprender programación."}

]
while True:
    entrada = input("tú> ")
    if entrada.strip().lower() in {"salir", "exit"}:
        break

    # Paso 1: agrega la entrada al historial como mensaje user.

    # Paso 2: llama a preguntar(messages) y guarda el resultado en respuesta.

    # Paso 3: imprime respuesta y agrégala al historial como mensaje assistant.

## Ejercicio 3: añade comandos

Mejora el chat del ejercicio anterior. Completa la celda para que:

- `salir` o `exit` termine el chat, aunque se escriba con mayúsculas.
- `reiniciar` borre el historial y empiece una conversación nueva conservando el mensaje `system`. Imprime `Conversación reiniciada.` y no llames a la API.
- Una entrada vacía o con solo espacios se ignore.
- Cualquier otro texto se envíe al asistente y se guarde en el historial.

**Prueba:** inicia un tema, escribe `reiniciar` y pregunta algo que dependa del tema anterior. ¿Lo recuerda?

In [ ]:
instruccion = {
    "role": "system",
    "content": "Eres un asistente breve y claro que ayuda a aprender programación.",
}
messages = [instruccion]

while True:
    entrada = input("tú> ")
    comando = entrada.strip().lower()

    if comando in {"salir", "exit"}:
        break

    if comando == "reiniciar":
        # Reinicia messages para que solo conserve instruccion.
        # Imprime un aviso y continúa con la siguiente vuelta del bucle.
        continue

    if not comando:
        # Ignora la entrada vacía y continúa con la siguiente vuelta.
        continue

    messages.append({"role": "user", "content": entrada})
    respuesta = preguntar(messages)
    print("bot>", respuesta)
    messages.append({"role": "assistant", "content": respuesta})